# 🕵️ Dia 05 — Cadeias de resolução DNS: seguindo um nome até um endereço

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 5* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-05.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

15h02. A TI da Serra Azul bloqueou `serraazul-pix.example` no resolvedor interno: de dentro, `NXDOMAIN`. Da VM da Vigia, pela internet: `203.0.113.77`. Rafa: *"Dois `dig`, duas respostas, as duas certas."*

E um cooperado jura que digitou o endereço **verdadeiro** e caiu na página falsa.

Neste notebook você vai:

1. **Ler a saída do `dig`** com Python e seguir a cadeia de CNAME até o A.
2. **Fazer uma consulta DNS real em JSON por HTTPS** (com plano B offline).
3. **Simular o cache**: duas respostas diferentes, as duas corretas.
4. **Cruzar o log do resolvedor com o `conn.log`** do Dia 4.
5. **Jogar o quiz "onde está a evidência?"** e gerar o diagrama da cadeia.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-05/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['dig-resolvedor-interno.txt'] = '; <<>> DiG 9.18.28 <<>> serraazul-pix.example\n;; global options: +cmd\n;; Got answer:\n;; ->>HEADER<<- opcode: QUERY, status: NXDOMAIN, id: 18842\n;; flags: qr rd ra; QUERY: 1, ANSWER: 0, AUTHORITY: 1, ADDITIONAL: 1\n\n;; QUESTION SECTION:\n;serraazul-pix.example.         IN      A\n\n;; AUTHORITY SECTION:\nrpz.serraazul.example.  60      IN      SOA     dns1.serraazul.example. ti.serraazul.example. 2026100503 3600 600 86400 60\n\n;; Query time: 1 msec\n;; SERVER: 10.20.32.53#53(10.20.32.53) (UDP)\n;; WHEN: Mon Oct 05 18:02:11 UTC 2026\n'
EMBUTIDO['dig-publico.txt'] = '; <<>> DiG 9.18.28 <<>> www.serraazul-pix.example @1.1.1.1\n;; global options: +cmd\n;; Got answer:\n;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 50317\n;; flags: qr rd ra; QUERY: 1, ANSWER: 2, AUTHORITY: 0, ADDITIONAL: 1\n\n;; QUESTION SECTION:\n;www.serraazul-pix.example.     IN      A\n\n;; ANSWER SECTION:\nwww.serraazul-pix.example. 3412 IN      CNAME   serraazul-pix.example.\nserraazul-pix.example.  187     IN      A       203.0.113.77\n\n;; Query time: 22 msec\n;; SERVER: 1.1.1.1#53(1.1.1.1) (UDP)\n;; WHEN: Mon Oct 05 18:02:40 UTC 2026\n'
EMBUTIDO['dig-trace.txt'] = '; <<>> DiG 9.18.28 <<>> +trace +nodnssec www.serraazul-pix.example\n.                       518400  IN      NS      a.root-servers.net.\n;; Received 239 bytes from 192.168.56.1#53(192.168.56.1) in 9 ms\n\nexample.                172800  IN      NS      a.tld-exemplo.example.\n;; Received 412 bytes from 198.41.0.4#53(a.root-servers.net) in 24 ms\n\nserraazul-pix.example.  86400   IN      NS      ns1.dnsbarato.example.\nserraazul-pix.example.  86400   IN      NS      ns2.dnsbarato.example.\n;; Received 301 bytes from 192.0.2.10#53(a.tld-exemplo.example) in 31 ms\n\nwww.serraazul-pix.example. 3600 IN      CNAME   serraazul-pix.example.\nserraazul-pix.example.  300     IN      A       203.0.113.77\n;; Received 102 bytes from 198.51.100.53#53(ns1.dnsbarato.example) in 88 ms\n'
EMBUTIDO['log-consultas-resolvedor.txt'] = '# Extrato do log de consultas do resolvedor interno dns1.serraazul.example (10.20.32.53)\n# Entregue pela TI da Serra Azul em 2026-10-05, filtrado pelo nome serraazul-pix.example. Horário do servidor: UTC.\n05-Oct-2026 11:14:02.117 client 10.20.36.21#53110: query: serraazul-pix.example IN A +E(0) (10.20.32.53)\n05-Oct-2026 11:31:45.390 client 10.20.36.44#61207: query: serraazul-pix.example IN A +E(0) (10.20.32.53)\n05-Oct-2026 12:02:16.902 client 10.20.36.44#61290: query: serraazul-pix.example IN A +E(0) (10.20.32.53)\n05-Oct-2026 14:47:33.188 client 10.20.36.21#53990: query: serraazul-pix.example IN A +E(0) (10.20.32.53)\n05-Oct-2026 14:52:08.870 client 10.20.36.30#51001: query: serraazul-pix.example IN A +E(0) (10.20.32.53)\n05-Oct-2026 17:40:00.000 rpz: zona rpz.serraazul.example recarregada, serial 2026100503 (bloqueio de serraazul-pix.example)\n05-Oct-2026 17:55:20.411 client 10.20.36.21#54102: query: serraazul-pix.example IN A +E(0) (10.20.32.53) -> rpz NXDOMAIN\n'
EMBUTIDO['hosts-didatico.txt'] = '# /etc/hosts — exemplo DIDÁTICO de pharming (fictício). Nenhuma consulta DNS sai da máquina.\n127.0.0.1       localhost\n::1             localhost\n203.0.113.77    serraazul.example\n203.0.113.77    www.serraazul.example\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import hashlib

TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
for nome, txt in TEXTOS.items():                      # Dia 2: hash de tudo o que chega
    print(f"{hashlib.sha256(txt.encode()).hexdigest()}  {nome}")

## 1. Lendo o `dig` como evidência

Um leitor simples da saída do `dig`: guarda o `status`, as *flags*, as seções e, sobretudo, a **proveniência** (`SERVER` e `WHEN`).

In [ ]:
import re

def ler_dig(texto):
    """Extrai de uma saída do dig: status, flags, registros por seção, SERVER e WHEN."""
    r = {"status": None, "flags": [], "secoes": {}, "server": None, "when": None}
    secao = None
    for linha in texto.splitlines():
        if m := re.search(r"status: (\w+)", linha):            # ->>HEADER<<- ... status: NOERROR
            r["status"] = m.group(1)
        if m := re.search(r"flags: ([a-z ]+);", linha):          # ;; flags: qr rd ra;
            r["flags"] = m.group(1).split()
        if m := re.match(r";; (\w+) SECTION:", linha):          # início de uma seção
            secao = m.group(1); r["secoes"][secao] = []; continue
        if linha.startswith(";; SERVER:"):
            r["server"] = linha.split(":", 1)[1].strip()
        if linha.startswith(";; WHEN:"):
            r["when"] = linha.split(":", 1)[1].strip()
        if secao and linha and not linha.startswith(";"):      # um registro: nome TTL classe tipo dado
            nome, ttl, _classe, tipo, *dado = linha.split()
            r["secoes"][secao].append({"nome": nome, "ttl": int(ttl), "tipo": tipo, "dado": " ".join(dado)})
        elif not linha.strip():
            secao = None                                       # linha em branco fecha a seção
    return r

for arq in ("dig-resolvedor-interno.txt", "dig-publico.txt"):
    d = ler_dig(TEXTOS[arq])
    print(f"== {arq}")
    print(f"   status={d['status']}  flags={d['flags']}  autoritativa={'aa' in d['flags']}")
    print(f"   SERVER={d['server']}  WHEN={d['when']}")
    for sec, regs in d["secoes"].items():
        if sec == "QUESTION":
            continue
        for x in regs:
            print(f"   [{sec}] {x['nome']} TTL={x['ttl']} {x['tipo']} {x['dado']}")

> 💡 Olhe a seção AUTHORITY do resolvedor interno: a zona se chama `rpz.serraazul.example`. RPZ (*Response Policy Zone*) é um mecanismo em que o resolvedor **reescreve** respostas por política. O `NXDOMAIN` é verdadeiro **para quem pergunta a esse resolvedor**. Não significa que o domínio deixou de existir.

## 2. Seguindo a cadeia de CNAME

In [ ]:
def seguir_cadeia(nome, registros):
    """Segue CNAMEs até achar um A/AAAA. Devolve a lista de saltos (nome, tipo, dado, ttl)."""
    saltos, visto = [], set()
    while nome not in visto:                             # proteção contra laço de CNAME
        visto.add(nome)
        r = next((x for x in registros if x["nome"] == nome), None)
        if r is None:
            break                                        # cadeia incompleta: falta consultar
        saltos.append((r["nome"], r["tipo"], r["dado"], r["ttl"]))
        if r["tipo"] != "CNAME":
            break                                        # chegamos ao endereço
        nome = r["dado"]                                 # o apelido aponta para o próximo nome
    return saltos

pub = ler_dig(TEXTOS["dig-publico.txt"])
for nome, tipo, dado, ttl in seguir_cadeia("www.serraazul-pix.example.", pub["secoes"]["ANSWER"]):
    print(f"{nome:28} --{tipo:5}--> {dado:24} (TTL {ttl})")

### O `+trace`: quem respondeu cada pedaço

Cada `;; Received ... from` nomeia o servidor que deu aquele bloco. Compare o TTL do registro A no autoritativo (valor configurado) com o do `dig` público (contagem regressiva de um cache).

In [ ]:
bloco = []
for linha in TEXTOS["dig-trace.txt"].splitlines():
    if linha.startswith(";; Received"):
        origem = re.search(r"from ([^ ]+)", linha).group(1)       # ip#porta(nome)
        print(f"🛰️  {origem}")
        for b in bloco:
            print("      ", " ".join(b.split()))
        bloco = []
    elif linha and not linha.startswith(";"):
        bloco.append(linha)

ttl_aut = 300                                                       # do último bloco do trace
ttl_cache = [x["ttl"] for x in pub["secoes"]["ANSWER"] if x["tipo"] == "A"][0]
print(f"\nTTL no autoritativo: {ttl_aut}s · no cache do 1.1.1.1: {ttl_cache}s → "
      f"a resposta pública estava em cache havia ~{ttl_aut - ttl_cache}s")

## 3. Uma consulta real (DoH em JSON), com plano B

Agora uma consulta **de verdade**, a um domínio público (o `.example` não resolve na internet). Usamos o resolvedor do Google em JSON por HTTPS. Sem internet, o notebook usa uma resposta de exemplo e avisa.

In [ ]:
#@title Consulta DNS por HTTPS { display-mode: "form" }
NOME = "www.wikipedia.org"  #@param {type:"string"}
TIPO = "A"  #@param ["A", "AAAA", "CNAME", "NS", "MX", "TXT"]

import json, urllib.request, urllib.parse
from datetime import datetime, timezone

url = "https://dns.google/resolve?" + urllib.parse.urlencode({"name": NOME, "type": TIPO})
quando = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")      # o WHEN da sua coleta
try:
    with urllib.request.urlopen(url, timeout=6) as r:
        resp = json.load(r)
    fonte = f"dns.google (DoH), {quando}"
except Exception as erro:
    # Plano B: resposta de exemplo no mesmo formato (valores ilustrativos do original do Dia 5)
    resp = {"Status": 0, "Answer": [
        {"name": "www.wikipedia.org.", "type": 5, "TTL": 49518, "data": "dyna.wikimedia.org."},
        {"name": "dyna.wikimedia.org.", "type": 1, "TTL": 300, "data": "208.80.154.224"}]}
    fonte = f"EXEMPLO OFFLINE (sem rede: {type(erro).__name__}) — não use como evidência"

TIPOS = {1: "A", 5: "CNAME", 28: "AAAA", 2: "NS", 15: "MX", 16: "TXT"}
print("Proveniência:", fonte)
print("Status:", {0: "NOERROR", 2: "SERVFAIL", 3: "NXDOMAIN"}.get(resp["Status"], resp["Status"]))
for a in resp.get("Answer", []):
    print(f"  {a['name']:28} TTL={a['TTL']:<6} {TIPOS.get(a['type'], a['type']):5} {a['data']}")

## 4. Simulando o cache: duas verdades

Dois investigadores perguntam o mesmo nome com dez minutos de diferença. Entre as duas perguntas, o operador do golpe troca o IP (TTL de 300 s). Os dois anotam IPs diferentes, e **nenhum está errado**.

In [ ]:
from datetime import timedelta

# A "zona" do autoritativo ao longo do tempo: (a partir de quando, IP). Fictício.
historico = [(datetime(2026, 10, 5, 18, 0, tzinfo=timezone.utc), "203.0.113.77"),
             (datetime(2026, 10, 5, 18, 7, tzinfo=timezone.utc), "203.0.113.79")]
TTL = 300

class Resolvedor:
    """Resolvedor recursivo de brinquedo: guarda a resposta em cache até o TTL vencer."""
    def __init__(self):
        self.cache = None                                   # (ip, vence_em)
    def perguntar(self, agora):
        if self.cache and agora < self.cache[1]:
            ip, vence = self.cache
            return ip, int((vence - agora).total_seconds()), "cache"
        ip = [ip for t, ip in historico if t <= agora][-1]  # pergunta ao autoritativo
        self.cache = (ip, agora + timedelta(seconds=TTL))
        return ip, TTL, "autoritativo"

r = Resolvedor()
for quem, minuto in [("Você", 2), ("Rafa", 5), ("Você", 8), ("Rafa", 12)]:
    agora = datetime(2026, 10, 5, 18, minuto, tzinfo=timezone.utc)
    ip, ttl, origem = r.perguntar(agora)
    print(f"{agora:%H:%M}Z  {quem:5} → {ip}  (TTL restante {ttl:>3}s, veio do {origem})")

## 5. Quem perguntou primeiro: log do resolvedor × `conn.log`

O log do resolvedor existe **antes** de qualquer conexão e sobrevive mesmo quando a conexão foi bloqueada. Vamos juntar os dois numa linha do tempo, em UTC e BRT.

In [ ]:
import urllib.request

# O conn.log é o do Dia 4 (mesmo caso). Mesma lógica de plano B.
CONN_EMBUTIDO = '#separator \\x09\n#set_separator\t,\n#empty_field\t(empty)\n#unset_field\t-\n#path\tconn\n#open\t2026-10-05-15-10-00\n#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\thistory\n#types\ttime\tstring\taddr\tport\taddr\tport\tenum\tstring\tinterval\tcount\tcount\tstring\tstring\n1791198842.118000\tCsA1kq3\t10.20.36.21\t53110\t10.20.32.53\t53\tudp\tdns\t0.002\t39\t55\tSF\tDd\n1791198842.131000\tCsA1kq4\t10.20.36.21\t50214\t203.0.113.77\t443\ttcp\tssl\t48.912\t1873\t41250\tSF\tShADadFf\n1791199905.402000\tCsA1kq9\t10.20.36.44\t50388\t203.0.113.77\t443\ttcp\tssl\t61.004\t2290\t41377\tSF\tShADadFf\n1791201737.006000\tCsA1kr2\t10.20.36.44\t50412\t203.0.113.77\t443\ttcp\tssl\t0.611\t517\t0\tRSTR\tShADr\n1791205791.550000\tCsA1kt0\t198.51.100.14\t41022\t10.20.32.10\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791205791.601000\tCsA1kt1\t198.51.100.14\t41024\t10.20.32.11\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791211653.200000\tCsA1ku5\t10.20.36.21\t50990\t203.0.113.77\t443\ttcp\t-\t9.012\t0\t0\tS0\tS\n1791211928.900000\tCsA1ku8\t10.20.36.30\t51007\t203.0.113.77\t443\ttcp\t-\t9.008\t0\t0\tS0\tS\n1791212620.000000\tCsA1kv0\t10.20.36.30\t51102\t10.20.32.80\t443\ttcp\tssl\t0.402\t1210\t5320\tSF\tShADadFf\n#close\t2026-10-05-15-10-00\n'
try:
    with urllib.request.urlopen(BASE.replace("dia-05", "dia-04") + "conn.log", timeout=5) as r:
        conn_txt = r.read().decode()
except Exception:
    conn_txt = CONN_EMBUTIDO

BRT = timezone(timedelta(hours=-3))
eventos = []
for l in TEXTOS["log-consultas-resolvedor.txt"].splitlines():
    if l.startswith("#"):
        continue
    t = datetime.strptime(l[:24], "%d-%b-%Y %H:%M:%S.%f").replace(tzinfo=timezone.utc)
    m = re.search(r"client ([\d.]+)", l)
    if m:
        bloq = "NXDOMAIN (RPZ)" if "rpz NXDOMAIN" in l else "resolvido"
        eventos.append((t, m.group(1), f"DNS  pergunta serraazul-pix.example → {bloq}"))
    else:
        eventos.append((t, "dns1", "DNS  " + l[25:].strip()))

campos = None
for l in conn_txt.splitlines():
    if l.startswith("#fields"):
        campos = l.split("\t")[1:]
    elif l and not l.startswith("#"):
        c = dict(zip(campos, l.split("\t")))
        if c["id.resp_h"] == "203.0.113.77":
            t = datetime.fromtimestamp(float(c["ts"]), timezone.utc)
            eventos.append((t, c["id.orig_h"], f"CONN → 203.0.113.77:443 {c['conn_state']} {c['history']}"))

for t, quem, o_que in sorted(eventos):
    print(f"{t:%H:%M:%S}Z  {t.astimezone(BRT):%H:%M:%S} BRT  {quem:12} {o_que}")

**Leia a linha do tempo e responda:** toda conexão teve uma consulta antes? A consulta das 12h02 UTC virou o quê? As conexões `S0` das 14h47 e 14h52 UTC aconteceram **antes** do bloqueio de DNS (17h40 UTC): o DNS respondeu, a conexão não. Então quem bloqueou essas conexões? (Guarde a hipótese para o Dia 9.)

## 6. O print do cooperado: arquivo *hosts*

In [ ]:
print(TEXTOS["hosts-didatico.txt"])

hosts = {}
for linha in TEXTOS["hosts-didatico.txt"].splitlines():
    linha = linha.split("#")[0].strip()                  # tira comentários
    if linha:
        ip, *nomes = linha.split()
        for n in nomes:
            hosts[n] = ip                                 # nome → IP, sem perguntar a ninguém

def resolver_como_o_sistema(nome):
    """O stub resolver olha o hosts ANTES do DNS. Se achar, nenhum pacote DNS sai da máquina."""
    if nome in hosts:
        return hosts[nome], "arquivo hosts (nenhuma consulta DNS saiu da máquina)"
    return "?", "perguntaria ao resolvedor recursivo (e ficaria no log dele)"

for nome in ["serraazul.example", "www.serraazul.example", "outro.example"]:
    print(nome, "→", *resolver_como_o_sistema(nome))

## 7. 🎮 Quiz: onde está a evidência?

Para cada pergunta do caso, escolha a fonte que guarda a resposta: **1** log do resolvedor recursivo · **2** `conn.log`/firewall · **3** arquivo *hosts* da máquina · **4** servidor autoritativo / `+trace`.

In [ ]:
FONTES = {"1": "log do resolvedor recursivo", "2": "conn.log / firewall",
          "3": "arquivo hosts da máquina", "4": "servidor autoritativo / +trace"}
QUIZ = [
    ("Qual estação interna perguntou primeiro por serraazul-pix.example?", "1",
     "O resolvedor registra cliente e hora, mesmo que a conexão nunca aconteça."),
    ("O cooperado digitou o endereço certo e caiu na página falsa. Onde procurar?", "3",
     "Pharming local: o hosts responde antes do DNS e não deixa rastro na rede."),
    ("Quais servidores de nomes respondem oficialmente pelo domínio do golpe?", "4",
     "A delegação (NS) e o TTL configurado vêm do autoritativo."),
    ("A estação abriu de fato uma conexão TCP com 203.0.113.77, e quanto trafegou?", "2",
     "DNS só mostra a pergunta. A conexão está no log de conexão."),
    ("Uma estação perguntou o nome, mas o firewall bloqueou a conexão. A pergunta ficou registrada?", "1",
     "Sim: a consulta existe antes da conexão e sobrevive ao bloqueio."),
    ("Qual o TTL configurado do registro A, sem contagem regressiva de cache?", "4",
     "Pergunte ao autoritativo com +norecurse: flag aa, TTL cheio."),
]
MODO_JOGO = True

acertos = 0
print("\n".join(f"{k} = {v}" for k, v in FONTES.items()))
for i, (perg, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {perg}")
    if MODO_JOGO:
        r = ""
        while r not in FONTES:
            r = input("   Fonte (1-4): ").strip()
        acertos += (r == certo)
        print("   ✅ Acertou!" if r == certo else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {FONTES[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 5 else "— meta: 5 de 6.")

## 8. 📝 O diagrama da cadeia (Passo 7) em mermaid

Gerado com os valores **dos arquivos**, com a nota de onde fica o log e a proveniência. Cole em `day05-chain-isca.md`.

In [ ]:
from pathlib import Path

trace = TEXTOS["dig-trace.txt"]
pub = ler_dig(TEXTOS["dig-publico.txt"])
diagrama = f"""flowchart LR
    DEV["VM de análise vigia-lab01<br/>stub resolver<br/>hosts conferido: sim"]
    REC["Resolvedor recursivo<br/>{pub['server']}"]
    ROOT["Servidor raiz<br/>a.root-servers.net<br/>198.41.0.4"]
    TLD["Servidor do TLD (.example)<br/>a.tld-exemplo.example<br/>192.0.2.10"]
    AUTH["Servidor autoritativo<br/>ns1.dnsbarato.example<br/>198.51.100.53"]
    CN["Salto de CNAME<br/>www.serraazul-pix.example para serraazul-pix.example<br/>TTL 3600"]
    AREC["Registro A<br/>serraazul-pix.example 203.0.113.77<br/>TTL 300"]
    NOTE["Na Serra Azul, o log do resolvedor 10.20.32.53 registra quem perguntou, e quando.<br/>Coletado em {pub['when']} via {pub['server']}"]
    DEV -->|"A? www.serraazul-pix.example<br/>veio: CNAME + A"| REC
    REC -->|"veio: indicação dos NS do .example"| ROOT
    ROOT -->|"veio: indicação de ns1/ns2.dnsbarato.example"| TLD
    TLD -->|"veio: CNAME + A"| AUTH
    AUTH --> CN
    CN --> AREC
    REC -.- NOTE"""
Path("notas").mkdir(exist_ok=True)
Path("notas/day05-chain-isca.md").write_text("```mermaid\n" + diagrama + "\n```\n", encoding="utf-8")
print(diagrama)
print("\n💾 notas/day05-chain-isca.md")
try:
    from google.colab import files
    files.download("notas/day05-chain-isca.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou na pasta notas/)")

## ✅ Antes de fechar o Dia 5

- [ ] Sabe explicar por que `NXDOMAIN` interno e `NOERROR` público são ambos verdadeiros (RPZ)
- [ ] Linha do tempo DNS + conexões revisada, com a consulta das 12h02 UTC explicada
- [ ] Explica *pharming* por arquivo *hosts* para a Marta, sem jargão
- [ ] Quiz com pelo menos 5/6
- [ ] Fez também a prática original com `www.wikipedia.org` (ou outra organização pública) e o DNSViz

**Amanhã (Dia 6):** o resto do DNS. O domínio verdadeiro da cooperativa tem DMARC `p=none`; o do golpe deixou um *token* de verificação no TXT. Hora de montar o grafo de pegada.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>